In [15]:
import lightgbm as lgb
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
import pandas as pd

## Data Exploration

In [16]:
df = pd.read_csv("data/NYC.csv")

df_shape = df.shape
df_height = df_shape[0]
df_width = df_shape[1]

cols = df.columns.tolist()

num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

print(f"Dataset Shape: {df_shape}")
print(f"Rows: {df_height}")
print(f"Columns: {df_width}")
print("\nNumeric Columns:")
print(num_cols)
print("\nCategorical Columns:")
print(cat_cols)
print("\nFirst 5 Rows:")

Dataset Shape: (1458644, 11)
Rows: 1458644
Columns: 11

Numeric Columns:
['vendor_id', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'trip_duration']

Categorical Columns:
['id', 'pickup_datetime', 'dropoff_datetime', 'store_and_fwd_flag']

First 5 Rows:


/tmp/ipykernel_18388/3940748691.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object"]).columns.tolist()


In [17]:
print(df.head())

          id  vendor_id      pickup_datetime     dropoff_datetime  \
0  id2875421          2  2016-03-14 17:24:55  2016-03-14 17:32:30   
1  id2377394          1  2016-06-12 00:43:35  2016-06-12 00:54:38   
2  id3858529          2  2016-01-19 11:35:24  2016-01-19 12:10:48   
3  id3504673          2  2016-04-06 19:32:31  2016-04-06 19:39:40   
4  id2181028          2  2016-03-26 13:30:55  2016-03-26 13:38:10   

   passenger_count  pickup_longitude  pickup_latitude  dropoff_longitude  \
0                1        -73.982155        40.767937         -73.964630   
1                1        -73.980415        40.738564         -73.999481   
2                1        -73.979027        40.763939         -74.005333   
3                1        -74.010040        40.719971         -74.012268   
4                1        -73.973053        40.793209         -73.972923   

   dropoff_latitude store_and_fwd_flag  trip_duration  
0         40.765602                  N            455  
1         40.731

In [18]:
print(f"Null (%) for each column:\n{(df.isnull().mean()*100).round(2)}")

Null (%) for each column:
id                    0.0
vendor_id             0.0
pickup_datetime       0.0
dropoff_datetime      0.0
passenger_count       0.0
pickup_longitude      0.0
pickup_latitude       0.0
dropoff_longitude     0.0
dropoff_latitude      0.0
store_and_fwd_flag    0.0
trip_duration         0.0
dtype: float64


## Data Processing

Xóa ID

In [19]:
df.drop("id", axis=1, inplace=True)

Feature Engineering từ pickup_datetime và dropoff_datetime

In [20]:
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])
df["dropoff_datetime"] = pd.to_datetime(df["dropoff_datetime"])

df["pickup_hour"] = df["pickup_datetime"].dt.hour
df["pickup_dayofweek"] = df["pickup_datetime"].dt.dayofweek
df["pickup_month"] = df["pickup_datetime"].dt.month
df["duration_minutes"] = (df["dropoff_datetime"] - df["pickup_datetime"]).dt.total_seconds() / 60

df.drop(["pickup_datetime", "dropoff_datetime"], axis=1, inplace=True)

Encode Categorical

In [21]:
categorical_cols = df.select_dtypes(
    include=["object"]
).columns

df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

/tmp/ipykernel_18388/1892738396.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(


Tách X và y

In [22]:
df = df.astype(float)

X = df.drop("trip_duration", axis=1).values
y = df["trip_duration"].values

## Huấn luyện LightGBM

Cross Validation

In [23]:
def kfold(y, k=5, random_state=42):
    np.random.seed(random_state)
    indices = np.arange(len(y))
    np.random.shuffle(indices)
    folds = np.array_split(indices, k)

    result = []
    for i in range(k):
        test_idx = folds[i]
        train_idx = np.concatenate(
            [folds[j] for j in range(k) if j != i]
        )
        result.append((train_idx, test_idx))

    return result

In [24]:
folds = kfold(y, k=5)

In [25]:
rmse_scores = []
mae_scores = []
r2_scores = []
mape_scores = []

Regression Metrics

In [26]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1))) * 100

In [27]:
for train_idx, test_idx in folds:

    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    model = lgb.LGBMRegressor(
        n_estimators=100,
        learning_rate=0.1,
        num_leaves=31,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse_scores.append(
        rmse(y_test, y_pred)
    )

    mae_scores.append(
        mae(y_test, y_pred)
    )

    r2_scores.append(
        r2(y_test, y_pred)
    )

    mape_scores.append(
        mape(y_test, y_pred)
    )

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.028342 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1326
[LightGBM] [Info] Number of data points in the train set: 1166915, number of used features: 11
[LightGBM] [Info] Start training from score 959.273585
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.030971 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1326
[LightGBM] [Info] Number of data points in the train set: 1166915, number of used features: 11
[LightGBM] [Info] Start training from score 965.094834
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022298 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is n

Kết quả

In [28]:
print("Average RMSE :", np.mean(rmse_scores))
print("Average MAE  :", np.mean(mae_scores))
print("Average R²   :", np.mean(r2_scores))
print("Average MAPE :", np.mean(mape_scores))

Average RMSE : 3653.1829179764345
Average MAE  : 69.60352437182952
Average R²   : 0.47918607784488565
Average MAPE : 6.187690011343241


## Huấn luyện với LightGBMLibrary (Custom)

In [29]:
import sys, os, time

PROJECT_ROOT = os.path.abspath("LightGBMLibrary")
sys.path.insert(0, PROJECT_ROOT)

from lgbm_numpy.api import LGBMRegressor as CustomLGBMRegressor
from lgbm_numpy.metrics import rmse as custom_rmse, mae as custom_mae

Dùng subset dữ liệu vì custom library tính bằng numpy thuần, sẽ rất chậm trên toàn bộ 1.4M rows

In [30]:
SUBSAMPLE_SIZE = 50000
np.random.seed(42)
subset_idx = np.random.choice(len(X), size=SUBSAMPLE_SIZE, replace=False)
X_sub = X[subset_idx].copy()
y_sub = y[subset_idx].copy()

X_mean = X_sub.mean(axis=0)
X_std = X_sub.std(axis=0)
X_std[X_std == 0] = 1.0
X_sub_scaled = (X_sub - X_mean) / X_std

print(f"Subset: {X_sub_scaled.shape[0]} samples, {X_sub_scaled.shape[1]} features")

Subset: 50000 samples, 11 features


Cross Validation với LightGBMLibrary

In [31]:
folds_sub = kfold(y_sub, k=5)

In [32]:
custom_rmse_scores = []
custom_mae_scores = []
custom_r2_scores = []
custom_mape_scores = []
custom_times = []

In [33]:
for train_idx, test_idx in folds_sub:

    X_train = X_sub_scaled[train_idx]
    X_test = X_sub_scaled[test_idx]

    y_train = y_sub[train_idx]
    y_test = y_sub[test_idx]

    model = CustomLGBMRegressor(
        n_estimators=100,
        learning_rate=0.1,
        num_leaves=31,
        min_data_in_leaf=20,
        lambda_l2=1.0,
        max_bin=63,
        verbose=False,
        seed=42
    )

    t0 = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - t0
    custom_times.append(elapsed)

    y_pred = model.predict(X_test)

    custom_rmse_scores.append(custom_rmse(y_test, y_pred))
    custom_mae_scores.append(custom_mae(y_test, y_pred))

    ss_res = np.sum((y_test - y_pred) ** 2)
    ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
    custom_r2_scores.append(1 - (ss_res / ss_tot))

    custom_mape_scores.append(
        np.mean(np.abs((y_test - y_pred) / np.maximum(np.abs(y_test), 1))) * 100
    )

    print(f"  Fold {len(custom_rmse_scores)}: RMSE={custom_rmse_scores[-1]:.2f}  "
          f"MAE={custom_mae_scores[-1]:.2f}  R²={custom_r2_scores[-1]:.4f}  ({elapsed:.1f}s)")

  Fold 1: RMSE=2434.64  MAE=179.62  R²=0.4250  (10.1s)
  Fold 2: RMSE=2888.36  MAE=228.15  R²=0.3821  (10.1s)
  Fold 3: RMSE=2532.64  MAE=189.15  R²=0.5503  (10.0s)
  Fold 4: RMSE=2572.29  MAE=186.84  R²=0.5305  (10.1s)
  Fold 5: RMSE=2142.62  MAE=179.31  R²=0.3400  (9.8s)


Kết quả LightGBMLibrary

In [34]:
print("Average RMSE :", np.mean(custom_rmse_scores))
print("Average MAE  :", np.mean(custom_mae_scores))
print("Average R²   :", np.mean(custom_r2_scores))
print("Average MAPE :", np.mean(custom_mape_scores))
print("Average Time :", np.mean(custom_times), "s")

Average RMSE : 2514.1098885138335
Average MAE  : 192.6140166502987
Average R²   : 0.4455717822426936
Average MAPE : 8.154461063275296
Average Time : 10.01225152015686 s


## So sánh hai thư viện

In [35]:
official_rmse = np.mean(rmse_scores)
official_mae = np.mean(mae_scores)
official_r2 = np.mean(r2_scores)
official_mape = np.mean(mape_scores)

custom_avg_rmse = np.mean(custom_rmse_scores)
custom_avg_mae = np.mean(custom_mae_scores)
custom_avg_r2 = np.mean(custom_r2_scores)
custom_avg_mape = np.mean(custom_mape_scores)

print(f"{'Metric':<12s} {'Official LightGBM':>18s} {'Custom Library':>18s} {'Difference':>12s}")
print("-" * 62)
print(f"{'RMSE':<12s} {official_rmse:>18.2f} {custom_avg_rmse:>18.2f} {custom_avg_rmse - official_rmse:>+12.2f}")
print(f"{'MAE':<12s} {official_mae:>18.2f} {custom_avg_mae:>18.2f} {custom_avg_mae - official_mae:>+12.2f}")
print(f"{'R²':<12s} {official_r2:>18.4f} {custom_avg_r2:>18.4f} {custom_avg_r2 - official_r2:>+12.4f}")
print(f"{'MAPE (%)':<12s} {official_mape:>18.2f} {custom_avg_mape:>18.2f} {custom_avg_mape - official_mape:>+12.2f}")

Metric        Official LightGBM     Custom Library   Difference
--------------------------------------------------------------
RMSE                    3653.18            2514.11     -1139.07
MAE                       69.60             192.61      +123.01
R²                       0.4792             0.4456      -0.0336
MAPE (%)                   6.19               8.15        +1.97
